# PDX bootstrap study

This notebook fits ten independent cell-bootstrap and ten paired sample-bootstrap models. Cell bootstraps resample human and mouse cells independently with replacement; sample bootstraps draw matched PDX samples with replacement. The validation cells, model initialization, and posterior seed remain fixed so the analysis isolates sampling variation.

PDX expression is rebuilt from raw counts in `.raw.X`: normalize each cell to 10,000 counts, then apply natural-log1p. The original counts are retained in `layers["counts"]`; saved expression and gene-selection statistics are not used.


In [ ]:
from itertools import combinations
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import spearmanr
import xenocomm as xc
from scipy import sparse

sns.set_theme(context="notebook", style="whitegrid")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "data").is_dir():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "notebooks"
DATA_DIR = NOTEBOOK_DIR / "data/melanoma_pdx_10k"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs/bootstrap_10k"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def read_pdx_counts(path):
    stored = ad.read_h5ad(path)
    if stored.raw is None:
        raise ValueError(f"{path} must contain raw counts in .raw.X")
    counts = sparse.csr_matrix(stored.raw.X, copy=True)
    if (
        not np.isfinite(counts.data).all()
        or np.any(counts.data < 0)
        or np.any(counts.data != np.floor(counts.data))
    ):
        raise ValueError(f"{path}: .raw.X must contain nonnegative integer counts")
    if np.any(np.asarray(counts.sum(axis=1)).ravel() <= 0):
        raise ValueError(f"{path}: every cell must have a positive count total")
    data = ad.AnnData(
        X=counts.astype(np.float32),
        obs=stored.obs[["sample", "cell_type"]].copy(),
        var=stored.raw.var[["gene_id"]].copy(),
    )
    data.layers["counts"] = counts
    sc.pp.normalize_total(data, target_sum=10_000)
    sc.pp.log1p(data)
    return data


adata_mouse = read_pdx_counts(DATA_DIR / "adata_mouse.h5ad")
adata_human = read_pdx_counts(DATA_DIR / "adata_human.h5ad")


In [ ]:
CELL_SEEDS = tuple(range(2026081701, 2026081711))
SAMPLE_SEEDS = tuple(range(2026081901, 2026081911))
TRAINING_SEED = 20260716
POSTERIOR_SEED = 20260717
STEPS_PER_BATCH = 250
EPOCHS = 5
POSTERIOR_SAMPLES = 2_000
VALIDATION_CELLS = 4096
VALIDATION_SPLIT_SEED = 20260717
VALIDATION_SEED = 20260718


def validation_indices(obs, size, seed):
    strata = obs[["sample", "cell_type"]].astype(str).agg("|".join, axis=1)
    counts = strata.value_counts().sort_index()
    exact = counts.to_numpy() * size / len(obs)
    quotas = np.floor(exact).astype(int)
    quotas[np.argsort(-(exact - quotas), kind="stable")[: size - quotas.sum()]] += 1
    rng = np.random.RandomState(seed)
    held_out = np.concatenate([
        rng.choice(np.flatnonzero(strata.to_numpy() == label), quota, replace=False)
        for label, quota in zip(counts.index, quotas, strict=True)
    ])
    rng.shuffle(held_out)
    return np.setdiff1d(np.arange(len(obs)), held_out), held_out


training_indices, held_out_indices = validation_indices(
    adata_mouse.obs, VALIDATION_CELLS, VALIDATION_SPLIT_SEED
)
source_mouse = adata_mouse[training_indices]
validation_mouse = adata_mouse[held_out_indices]
network = xc.prepare_network(adata_mouse, adata_human, dispersion_cutoff=-10)


In [ ]:
def cell_bootstrap(seed):
    mouse_stream, human_stream = np.random.SeedSequence(seed).spawn(2)
    mouse_rng = np.random.default_rng(mouse_stream)
    human_rng = np.random.default_rng(human_stream)
    mouse_indices = mouse_rng.integers(0, source_mouse.n_obs, source_mouse.n_obs)
    human_indices = human_rng.integers(0, adata_human.n_obs, adata_human.n_obs)
    return source_mouse[mouse_indices], adata_human[human_indices]


def sample_bootstrap(seed):
    mouse_samples = source_mouse.obs["sample"].astype(str).to_numpy()
    human_samples = adata_human.obs["sample"].astype(str).to_numpy()
    samples = tuple(sorted(set(mouse_samples)))
    if set(samples) != set(human_samples):
        raise ValueError("Human and mouse sample names must match")
    rng = np.random.default_rng(np.random.SeedSequence(seed).spawn(1)[0])
    draw = rng.integers(0, len(samples), len(samples))
    mouse_indices = np.concatenate([np.flatnonzero(mouse_samples == samples[index]) for index in draw])
    human_indices = np.concatenate([np.flatnonzero(human_samples == samples[index]) for index in draw])
    return source_mouse[mouse_indices], adata_human[human_indices]


In [ ]:
run_tables = []
receptor_tables = []
run_parameters = {}
for resampling, seeds, resample in (
    ("Cell", CELL_SEEDS, cell_bootstrap),
    ("Sample", SAMPLE_SEEDS, sample_bootstrap),
):
    for replicate, seed in enumerate(seeds, start=1):
        stem = f"{resampling.lower()}_{replicate:02d}"
        table_path = OUTPUT_DIR / f"{stem}_ligands.parquet"
        receptor_path = OUTPUT_DIR / f"{stem}_receptors.parquet"
        parameter_path = OUTPUT_DIR / f"{stem}_parameters.npz"
        if table_path.exists() and receptor_path.exists() and parameter_path.exists():
            table = pd.read_parquet(table_path)
            receptors = pd.read_parquet(receptor_path)
            with np.load(parameter_path, allow_pickle=False) as saved:
                parameters = {name: np.asarray(saved[name]) for name in saved.files}
        else:
            print(f"Training {resampling} bootstrap {replicate}")
            bootstrap_mouse, bootstrap_human = resample(seed)
            ligand_abundance = xc.compute_ligand_abundance(
                bootstrap_mouse,
                bootstrap_human,
                network["ligands"],
                network["human_ligands"],
                network["ligand_receptor_matrix"],
            )
            model = xc.XenocommModel(
                bootstrap_mouse,
                **network,
                mean_ligand=ligand_abundance,
                receptor_target_mode="learned",
                training_seed=TRAINING_SEED,
                posterior_seed=POSTERIOR_SEED,
                batch_size=1024,
                steps_per_batch=STEPS_PER_BATCH,
                epochs=EPOCHS,
            )
            model.train(
                validation_mouse=validation_mouse,
                absolute_tolerance=0.001 * 1024,
                patience=3,
                min_evaluations=5,
                validation_seed=VALIDATION_SEED,
            )
            samples = model.sample(POSTERIOR_SAMPLES)
            parameters = model.get_parameters()
            table = xc.ligand_result_table(
                model.ligands,
                samples,
                model.mean_ligand_np,
                model.ligand_receptor_matrix_np,
                xc.get_receptor_sensitivity(parameters),
            )
            receptors = xc.receptor_marginal_df(model, samples, parameters)
            receptors["total_activation"] = receptors["mouse"] + receptors["delta"]
            table.to_parquet(table_path, index=False)
            receptors.to_parquet(receptor_path, index=False)
            np.savez(parameter_path, **parameters)
        run_tables.append(table.assign(resampling=resampling, replicate=replicate, bootstrap_seed=seed))
        receptor_tables.append(receptors.assign(resampling=resampling, replicate=replicate, bootstrap_seed=seed))
        run_parameters[(resampling, replicate)] = parameters


In [ ]:
bootstrap_ligands = pd.concat(run_tables, ignore_index=True)
bootstrap_receptors = pd.concat(receptor_tables, ignore_index=True)
bootstrap_ligands.to_parquet(OUTPUT_DIR / "ligand_results.parquet", index=False)
bootstrap_receptors.to_parquet(OUTPUT_DIR / "receptor_results.parquet", index=False)

call_frequency = (
    bootstrap_ligands.groupby(["resampling", "ligand"])["called"]
    .mean()
    .rename("detection_frequency")
    .reset_index()
)
call_frequency.to_parquet(OUTPUT_DIR / "detection_frequency.parquet", index=False)
display(call_frequency.sort_values("detection_frequency", ascending=False).head(20))


In [ ]:
def effective_weights(parameters):
    weights = np.square(parameters["gamma"])
    return weights / np.clip(weights.sum(axis=1, keepdims=True), 1e-6, 1)


def transformed_parameters(parameters):
    return {
        "alpha": np.logaddexp(0, parameters["alpha"]),
        "beta": np.exp(parameters["beta"]),
        "gamma": effective_weights(parameters),
    }


stability_rows = []
for resampling in ("Cell", "Sample"):
    for left, right in combinations(range(1, 11), 2):
        left_parameters = transformed_parameters(run_parameters[(resampling, left)])
        right_parameters = transformed_parameters(run_parameters[(resampling, right)])
        for parameter in ("alpha", "beta"):
            stability_rows.append({
                "resampling": resampling,
                "pair": f"{left}-{right}",
                "parameter": parameter,
                "spearman": spearmanr(left_parameters[parameter], right_parameters[parameter]).statistic,
            })
        gamma_correlations = []
        for receptor_index, (left_row, right_row) in enumerate(
            zip(left_parameters["gamma"], right_parameters["gamma"], strict=True)
        ):
            connected = network["receptor_target_matrix"][receptor_index].astype(bool)
            if connected.sum() < 2:
                continue
            correlation = spearmanr(left_row[connected], right_row[connected]).statistic
            if np.isfinite(correlation):
                gamma_correlations.append(correlation)
        stability_rows.append({
            "resampling": resampling,
            "pair": f"{left}-{right}",
            "parameter": "gamma",
            "spearman": np.nanmedian(gamma_correlations),
        })
        left_ranks = bootstrap_ligands.query("resampling == @resampling and replicate == @left").set_index("ligand")["rank"]
        right_ranks = bootstrap_ligands.query("resampling == @resampling and replicate == @right").set_index("ligand")["rank"]
        stability_rows.append({
            "resampling": resampling,
            "pair": f"{left}-{right}",
            "parameter": "ligand rank",
            "spearman": spearmanr(left_ranks, right_ranks.loc[left_ranks.index]).statistic,
        })

stability = pd.DataFrame(stability_rows)
stability.to_parquet(OUTPUT_DIR / "stability.parquet", index=False)
stability.groupby(["resampling", "parameter"])["spearman"].median().unstack()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=stability.query("parameter != 'ligand rank'"), x="parameter", y="spearman", hue="resampling", ax=ax)
ax.set(xlabel="Parameter", ylabel="Pairwise Spearman correlation", title="Bootstrap parameter stability", ylim=(0, 1))
plt.tight_layout()


In [ ]:
top_ligands = bootstrap_ligands.groupby("ligand")["delta_h_mean"].mean().nlargest(6).index
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(
    data=bootstrap_ligands[bootstrap_ligands["ligand"].isin(top_ligands)],
    x="ligand",
    y="delta_h_mean",
    hue="resampling",
    order=top_ligands,
    ax=ax,
)
ax.set(xlabel="Ligand", ylabel="Human counterfactual activation", title="Ligand stability")
plt.tight_layout()


In [ ]:
top_receptors = bootstrap_receptors.groupby("receptor")["total_activation"].mean().nlargest(6).index
fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(
    data=bootstrap_receptors[bootstrap_receptors["receptor"].isin(top_receptors)],
    x="receptor",
    y="total_activation",
    hue="resampling",
    order=top_receptors,
    ax=ax,
)
ax.set(xlabel="Receptor", ylabel="Overall activation", title="Receptor activation stability")
plt.tight_layout()
